# 00 - Entorno y primer pipeline con PySpark

En este notebook vamos a conectar la teoria con una primera ejecucion local.

Al terminar seras capaz de:

- crear una `SparkSession`;
- identificar el modo de ejecucion y los recursos locales;
- crear y explorar un DataFrame;
- observar sus particiones;
- distinguir transformaciones de acciones;
- comprobar la evaluacion diferida de Spark.

> Dinamica: **predice -> ejecuta -> observa -> explica**.

## 1. Crear la SparkSession

La `SparkSession` es el punto de entrada principal a Spark. `local[*]` indica que ejecutaremos Spark en esta maquina y que puede utilizar todos los cores locales disponibles.

### Preflight de Java para el kernel

Si VS Code/Jupyter no hereda `JAVA_HOME`, ejecuta esta celda antes de crear la `SparkSession`.


In [1]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print("java path =", shutil.which("java"))
subprocess.run(["java", "-version"], check=True)


JAVA_HOME = /Users/beatrizsuch/java/temurin-17/Contents/Home
java path = /Users/beatrizsuch/java/temurin-17/Contents/Home/bin/java


openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("pontia-primer-pipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .enableHiveSupport()  # requiere entorno compatible con Hive
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/14 16:52:19 WARN Utils: Your hostname, MacBook-Pro-de-Beatriz.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.68 instead (on interface en0)
26/11/14 16:52:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 16:52:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(f"Spark: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Spark: 4.3.0
Master: local[*]
Paralelismo por defecto: 8
Spark UI: http://192.168.1.68:4040


### Relacion con la arquitectura

En un cluster real, el driver coordina executors distribuidos entre diferentes worker nodes. En modo local todo vive en nuestro ordenador, pero se conserva el mismo modelo mental: hay un driver, datos divididos en particiones y tasks que utilizan los cores disponibles.

## 2. Crear un DataFrame pequeño

Usamos datos pequenos para comprender el comportamiento. El objetivo todavia no es medir rendimiento.

In [4]:
trips_data = [
    (1, "Madrid", "Centro", 12.5, 2.0),
    (2, "Madrid", "Norte", 8.0, 0.0),
    (3, "Valencia", "Centro", 15.0, 3.0),
    (4, "Sevilla", "Sur", -2.0, 0.0),
    (5, "Madrid", "Centro", 21.0, 4.5),
    (6, "Valencia", "Norte", 11.0, 1.0),
    (7, "Sevilla", "Centro", 7.5, 0.5),
    (8, "Madrid", "Sur", 18.0, 2.5),
]

columns = ["trip_id", "city", "zone", "fare", "tip"]
trips = spark.createDataFrame(trips_data, columns)

trips.printSchema()
trips.show()

root
 |-- trip_id: long (nullable = true)
 |-- city: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- fare: double (nullable = true)
 |-- tip: double (nullable = true)



+-------+--------+------+----+---+
|trip_id|    city|  zone|fare|tip|
+-------+--------+------+----+---+
|      1|  Madrid|Centro|12.5|2.0|
|      2|  Madrid| Norte| 8.0|0.0|
|      3|Valencia|Centro|15.0|3.0|
|      4| Sevilla|   Sur|-2.0|0.0|
|      5|  Madrid|Centro|21.0|4.5|
|      6|Valencia| Norte|11.0|1.0|
|      7| Sevilla|Centro| 7.5|0.5|
|      8|  Madrid|   Sur|18.0|2.5|
+-------+--------+------+----+---+



## 3. Observar las particiones

Para que el reparto sea visible, crearemos cuatro particiones y anadiremos temporalmente su identificador.

Antes de ejecutar la celda, predice: ¿todas las particiones tendran exactamente el mismo numero de filas?

In [5]:
trips_4p = trips.repartition(4)

print(f"Numero de particiones: {trips_4p.rdd.getNumPartitions()}")

(
    trips_4p
    .withColumn("partition_id", F.spark_partition_id())
    .orderBy("partition_id", "trip_id")
    .show()
)

Numero de particiones: 4
+-------+--------+------+----+---+------------+
|trip_id|    city|  zone|fare|tip|partition_id|
+-------+--------+------+----+---+------------+
|      3|Valencia|Centro|15.0|3.0|           0|
|      6|Valencia| Norte|11.0|1.0|           0|
|      7| Sevilla|Centro| 7.5|0.5|           0|
|      1|  Madrid|Centro|12.5|2.0|           1|
|      2|  Madrid| Norte| 8.0|0.0|           1|
|      8|  Madrid|   Sur|18.0|2.5|           2|
|      4| Sevilla|   Sur|-2.0|0.0|           3|
|      5|  Madrid|Centro|21.0|4.5|           3|
+-------+--------+------+----+---+------------+



Una particion es una unidad de datos que puede ser procesada por una task dentro de un stage. Mas particiones aportan paralelismo potencial, pero los cores disponibles determinan cuantas tasks pueden ejecutarse simultaneamente.

## 4. Construir un pipeline de transformaciones

Las siguientes operaciones describen el resultado, pero no solicitan todavia que se devuelvan filas a Python.

In [6]:
valid_trips = (
    trips_4p
    .filter(F.col("fare") > 0)
    .select("city", "zone", "fare", "tip")
    .withColumn("total_amount", F.col("fare") + F.col("tip"))
)

summary_by_city = (
    valid_trips
    .groupBy("city")
    .agg(
        F.count("*").alias("number_of_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    )
)

print("Pipeline definido. Aun no hemos pedido sus filas.")

Pipeline definido. Aun no hemos pedido sus filas.


## 5. Inspeccionar el plan antes de ejecutar

`explain()` permite observar como Spark pretende ejecutar el pipeline. Busca `Scan`, `Filter`, `Project`, `HashAggregate` y `Exchange`. No es necesario comprender todavia todos los detalles.

In [7]:
summary_by_city.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- HashAggregate (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Project (5)
            +- Exchange (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [5]: [trip_id#0L, city#1, zone#2, fare#3, tip#4]
Arguments: [trip_id#0L, city#1, zone#2, fare#3, tip#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [5]: [trip_id#0L, city#1, zone#2, fare#3, tip#4]
Condition : (isnotnull(fare#3) AND (fare#3 > 0.0))

(3) Project
Output [3]: [city#1, fare#3, tip#4]
Input [5]: [trip_id#0L, city#1, zone#2, fare#3, tip#4]

(4) Exchange
Input [3]: [city#1, fare#3, tip#4]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=95]

(5) Project
Output [2]: [city#1, (fare#3 + tip#4) AS total_amount#41]
Input [3]: [city#1, fare#3, tip#4]

(6) HashAggregate
Input [2]: [

## 6. Una accion dispara la ejecucion

Predice primero el resultado. Despues ejecuta `show()` y observa la Spark UI si quieres comprobar el job generado.

In [8]:
summary_by_city.orderBy("city").show()

+--------+---------------+-------------+
|    city|number_of_trips|total_revenue|
+--------+---------------+-------------+
|  Madrid|              4|         68.5|
| Sevilla|              1|          8.0|
|Valencia|              2|         30.0|
+--------+---------------+-------------+



### Dos usos diferentes de `count`

- `groupBy(...).count()` devuelve otro DataFrame y continua construyendo el plan.
- `dataframe.count()` devuelve un entero a Python y es una accion.

In [9]:
counts_by_zone = valid_trips.groupBy("zone").count()  # transformacion agregada
number_of_rows = counts_by_zone.count()                  # accion

print(f"El DataFrame agregado contiene {number_of_rows} filas.")
counts_by_zone.orderBy("zone").show()

El DataFrame agregado contiene 3 filas.
+------+-----+
|  zone|count|
+------+-----+
|Centro|    4|
| Norte|    2|
|   Sur|    1|
+------+-----+



## 7. Configuraciones utiles (seleccion)

Estas configuraciones aparecen con frecuencia en proyectos reales. En este notebook no buscamos memorizarlas todas, sino entender **para que sirven** y **cuando tocarlas**.

- **Rendimiento SQL:** `spark.sql.shuffle.partitions`, `spark.sql.adaptive.enabled`, `spark.sql.autoBroadcastJoinThreshold`.
- **Recursos:** `spark.executor.memory`, `spark.driver.memory`, `spark.executor.cores`.
- **IO y formatos:** `spark.sql.parquet.compression.codec`, `spark.sql.sources.partitionOverwriteMode`.
- **Serializacion:** `spark.serializer=org.apache.spark.serializer.KryoSerializer`.
- **Pandas/Arrow:** `spark.sql.execution.arrow.pyspark.enabled=true`.
- **Dependencias/extensiones:** `spark.jars`, `spark.jars.packages`, `spark.sql.extensions`.

Puedes listar la configuracion activa con `spark.sparkContext.getConf().getAll()`.


### Importante

- Algunas configuraciones **pueden cambiarse en caliente** con `spark.conf.set(...)` (sobre todo varias de Spark SQL).
- Otras (memoria, cores, ciertos parametros del runtime) normalmente se fijan **al arrancar la aplicacion**.


In [13]:
conf_all = spark.sparkContext.getConf().getAll()
print('Nº de configs activas:', len(conf_all))
for k, v in sorted(conf_all):
    if k.startswith('spark.sql') or k in {
        'spark.master',
        'spark.app.name',
        'spark.driver.memory',
        'spark.executor.memory',
        'spark.executor.cores',
        'spark.serializer',
    }:
        print(f'{k} = {v}')


Nº de configs activas: 18
spark.app.name = pontia-primer-pipeline
spark.master = local[*]
spark.sql.artifact.isolation.enabled = false
spark.sql.catalogImplementation = hive
spark.sql.shuffle.partitions = 4
spark.sql.warehouse.dir = file:/Users/beatrizsuch/spark_project/spark-warehouse


## 9. Cerrar la sesion

Ejecuta esta celda al terminar. La Spark UI dejara de estar disponible cuando se cierre la sesion.

In [11]:
spark.stop()